# Offside Project Colab Test Notebook
This notebook is a self-contained copy of the currently implemented video, YOLO detection, ByteTrack tracking, shirt-color classification, JSON, and drawing workflow. It creates an annotated video for inspection. Calibration, 3D localization, pass detection, and the offside decision are not implemented in the repository yet, so this notebook does not claim an offside result. Start with a short test clip.

In [ ]:
%pip install -q ultralytics opencv-python-headless

In [ ]:
from collections import Counter, defaultdict, deque
from pathlib import Path
import json
import cv2
import numpy as np
from ultralytics import YOLO
from google.colab import files
from IPython.display import Video, display

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Upload a video file to continue')
VIDEO_PATH = next(iter(uploaded))
print('Input video:', VIDEO_PATH)

## Settings
Change the shirt color ranges to match your teams. HSV hue is 0 to 179 in OpenCV. The pretrained COCO model uses class 0 for person and class 32 for sports ball.

In [ ]:
MODEL_NAME = 'yolov8n.pt'  # Try yolov8s.pt for a larger model
PLAYER_CONF = 0.25
BALL_CONF = 0.10
IMGSZ = 960
PLAYER_CLASS_ID = 0
BALL_CLASS_ID = 32
TRACKER = 'bytetrack.yaml'  # Ultralytics' built-in ByteTrack configuration
MAX_FRAMES = None  # Set to e.g. 300 for a quick 10-second test at 30 FPS
OUTPUT_VIDEO = 'tracked_teams.mp4'
OUTPUT_JSON = 'detections_and_teams.json'

TEAM_HSV_RANGES = {
    'TEAM_A': [
        ((0, 70, 50), (10, 255, 255)),
        ((170, 70, 50), (179, 255, 255)),
    ],
    'TEAM_B': [
        ((90, 60, 40), (130, 255, 255)),
    ],
}
TEAM_COLORS = {
    'TEAM_A': (40, 40, 230),
    'TEAM_B': (230, 120, 40),
    'UNKNOWN': (180, 180, 180),
    'BALL': (0, 230, 255),
}

In [ ]:
def load_video_metadata(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f'Could not open video: {video_path}')
    try:
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = float(cap.get(cv2.CAP_PROP_FPS)) or 30.0
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        return {'path': str(video_path), 'width': width, 'height': height, 'fps': fps, 'frame_count': frame_count, 'duration_seconds': frame_count / fps}
    finally:
        cap.release()

def crop_jersey_region(frame, bbox_xyxy):
    height, width = frame.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in bbox_xyxy]
    x1, x2 = max(0, x1), min(width, x2)
    y1, y2 = max(0, y1), min(height, y2)
    box_w, box_h = x2 - x1, y2 - y1
    if box_w < 4 or box_h < 8:
        return None
    left, right = x1 + int(box_w * 0.2), x2 - int(box_w * 0.2)
    top, bottom = y1 + int(box_h * 0.2), y1 + int(box_h * 0.58)
    return frame[top:bottom, left:right] if right > left and bottom > top else None

def estimate_dominant_color(image):
    if image is None or image.size == 0:
        raise ValueError('Cannot estimate color from an empty image')
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    pixels = hsv.reshape(-1, 3)
    saturated = pixels[pixels[:, 1] >= 35]
    selected = saturated if len(saturated) else pixels
    return tuple(int(v) for v in np.median(selected, axis=0))

def classify_team(frame, bbox_xyxy, min_fraction=0.08):
    crop = crop_jersey_region(frame, bbox_xyxy)
    if crop is None or crop.size == 0:
        return 'UNKNOWN', 0.0
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    scores = {}
    for team, ranges in TEAM_HSV_RANGES.items():
        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
        for lower, upper in ranges:
            mask = cv2.bitwise_or(mask, cv2.inRange(hsv, np.array(lower, np.uint8), np.array(upper, np.uint8)))
        scores[team] = float(np.count_nonzero(mask)) / mask.size
    team, score = max(scores.items(), key=lambda item: item[1])
    return (team, score) if score >= min_fraction else ('UNKNOWN', score)

def draw_detections(frame, detections):
    output = frame.copy()
    for det in detections:
        x1, y1, x2, y2 = [int(round(v)) for v in det['bbox_xyxy']]
        label = det['label']
        team = 'BALL' if det['class_id'] == BALL_CLASS_ID else det.get('team', 'UNKNOWN')
        color = TEAM_COLORS.get(team, TEAM_COLORS['UNKNOWN'])
        cv2.rectangle(output, (x1, y1), (x2, y2), color, 2)
        track = '' if det.get('track_id') is None else f" ID:{det['track_id']}"
        text = f"{team} {label}{track} {det['confidence']:.2f}"
        cv2.putText(output, text, (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2, cv2.LINE_AA)
    return output

def save_json(data, path):
    with open(path, 'w', encoding='utf-8') as stream:
        json.dump(data, stream, indent=2, ensure_ascii=False)
    return path

def load_model(model_path=MODEL_NAME):
    return YOLO(model_path)

def detect_video(model, video_path, output_path=OUTPUT_VIDEO, json_path=OUTPUT_JSON):
    metadata = load_video_metadata(video_path)
    cap = cv2.VideoCapture(str(video_path))
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), metadata['fps'], (metadata['width'], metadata['height']))
    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f'Could not create output video: {output_path}')
    detections_by_frame = []
    team_votes = defaultdict(Counter)
    frame_index = 0
    try:
        while MAX_FRAMES is None or frame_index < MAX_FRAMES:
            ok, frame = cap.read()
            if not ok:
                break
            result = model.track(
                frame, persist=True, tracker=TRACKER,
                conf=min(PLAYER_CONF, BALL_CONF), imgsz=IMGSZ,
                classes=[PLAYER_CLASS_ID, BALL_CLASS_ID], verbose=False,
            )[0]
            frame_detections = []
            boxes = result.boxes
            if boxes is not None and len(boxes) > 0:
                xyxy = boxes.xyxy.cpu().numpy()
                class_ids = boxes.cls.cpu().numpy().astype(int)
                confidences = boxes.conf.cpu().numpy()
                ids = boxes.id.cpu().numpy().astype(int).tolist() if boxes.id is not None else [None] * len(boxes)
                for box, class_id, confidence, track_id in zip(xyxy, class_ids, confidences, ids):
                    threshold = PLAYER_CONF if class_id == PLAYER_CLASS_ID else BALL_CONF
                    if confidence < threshold:
                        continue
                    label = result.names.get(int(class_id), str(class_id))
                    det = {'frame_index': frame_index, 'class_id': int(class_id), 'label': str(label), 'confidence': float(confidence), 'bbox_xyxy': [float(v) for v in box], 'track_id': track_id}
                    if class_id == PLAYER_CLASS_ID:
                        team, score = classify_team(frame, box)
                        det['team_score'] = score
                        if track_id is not None and team != 'UNKNOWN':
                            team_votes[track_id][team] += 1
                            team = team_votes[track_id].most_common(1)[0][0]
                        det['team'] = team
                    frame_detections.append(det)
            detections_by_frame.append({'frame_index': frame_index, 'detections': frame_detections})
            writer.write(draw_detections(frame, frame_detections))
            frame_index += 1
            if frame_index % 100 == 0:
                print(f'Processed {frame_index} frames')
    finally:
        cap.release()
        writer.release()
    data = {'video_path': str(video_path), 'metadata': metadata, 'frames': detections_by_frame}
    save_json(data, json_path)
    return data

model = load_model()
results = detect_video(model, VIDEO_PATH)
print(f"Done: {len(results['frames'])} frames processed")
print('Annotated video:', OUTPUT_VIDEO)
print('Detection JSON:', OUTPUT_JSON)

In [ ]:
display(Video(OUTPUT_VIDEO, embed=True))
files.download(OUTPUT_VIDEO)
files.download(OUTPUT_JSON)